# Vision Agent Lab

This notebook reorganizes the `#6 비전 에이전트` lecture into a practical lab.

The lecture itself is app-oriented, so the notebook focuses on the vision logic behind those apps:
- frame handling and UI design ideas
- GrabCut segmentation logic
- color-based warning logic
- visual effects

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = Path("lab-session/05-vision-agent")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CAT_PATH = DATA_DIR / "cat.jpg"
OBJECT_PATH = DATA_DIR / "object.jpg"
VIEW_PATH = DATA_DIR / "view.jpg"
print("Base dir:", BASE_DIR.resolve())

## App design idea

A vision agent in this lecture is a small interactive program that combines:
- input source: webcam or image file
- vision logic: segmentation, color analysis, or filtering
- action: show an alert, save a result, or update a window

## GrabCut logic on a static image

In [ ]:
image = cv2.imread(str(OBJECT_PATH))
if image is None:
    raise FileNotFoundError(OBJECT_PATH)

mask = np.zeros(image.shape[:2], np.uint8)
bgd_model = np.zeros((1, 65), np.float64)
fgd_model = np.zeros((1, 65), np.float64)
rect = (40, 40, image.shape[1] - 80, image.shape[0] - 80)

cv2.grabCut(image, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
mask2 = np.where((mask == cv2.GC_BGD) | (mask == cv2.GC_PR_BGD), 0, 1).astype("uint8")
grabcut = image * mask2[:, :, np.newaxis]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(mask2, cmap="gray")
axes[1].set_title("Foreground mask")
axes[2].imshow(cv2.cvtColor(grabcut, cv2.COLOR_BGR2RGB))
axes[2].set_title("GrabCut result")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Color-based warning logic

In [ ]:
sample = cv2.imread(str(CAT_PATH))
if sample is None:
    raise FileNotFoundError(CAT_PATH)

hsv = cv2.cvtColor(sample, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, np.array([0, 80, 80]), np.array([20, 255, 255]))
mask |= cv2.inRange(hsv, np.array([160, 80, 80]), np.array([179, 255, 255]))
overlay = sample.copy()
overlay[mask > 0] = [0, 0, 255]
ratio = np.count_nonzero(mask) / mask.size

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
axes[0].set_title("Input")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Color mask")
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Overlay ({ratio:.2%})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Special effects pipeline

In [ ]:
view = cv2.imread(str(VIEW_PATH))
if view is None:
    raise FileNotFoundError(VIEW_PATH)
gray = cv2.cvtColor(view, cv2.COLOR_BGR2GRAY)

sketch_blur = cv2.GaussianBlur(gray, (21, 21), 0)
sketch = cv2.divide(gray, 255 - sketch_blur, scale=256)
edges = cv2.Canny(gray, 80, 160)
color = cv2.bilateralFilter(view, 9, 75, 75)
edge_mask = cv2.adaptiveThreshold(
    cv2.medianBlur(gray, 7),
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    9,
    5,
)
cartoon = cv2.bitwise_and(color, color, mask=edge_mask)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Original")
axes[0, 1].imshow(sketch, cmap="gray")
axes[0, 1].set_title("Sketch")
axes[1, 0].imshow(edges, cmap="gray")
axes[1, 0].set_title("Edges")
axes[1, 1].imshow(cv2.cvtColor(cartoon, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("Cartoon")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()

## App mapping

Use the scripts for the full desktop experience:
- `python examples/01_pyqt_control_panel.py`
- `python examples/02_pyqt_video_capture.py`
- `python examples/03_pyqt_grabcut_agent.py`
- `python examples/04_pyqt_color_zone_alert.py`
- `python examples/05_pyqt_photo_effects.py`